In [1]:


import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
cancer = load_breast_cancer()

X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

print("Dataset Loaded Successfully!")
print("\nFirst 5 Rows:")
print(X.head())

print("\nDataset Shape:", X.shape)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5,
    random_state=42, stratify=y_temp
)

print("\nTraining Samples:", len(X_train))
print("Validation Samples:", len(X_val))
print("Testing Samples:", len(X_test))
baseline_model = LogisticRegression(max_iter=5000)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test, baseline_predictions
)

print("\nBASELINE MODEL")
print("Algorithm: Logistic Regression")
print("Baseline Accuracy:", round(baseline_accuracy * 100, 2), "%")
cv_scores = cross_val_score(
    baseline_model, X_train, y_train, cv=5, scoring='accuracy'
)

print("\n5-FOLD CROSS VALIDATION")
print("Individual Scores:", cv_scores)
print("Cross Validation Mean:", round(cv_scores.mean() * 100, 2), "%")
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=5000))
])

pipeline.fit(X_train, y_train)

pipeline_predictions = pipeline.predict(X_val)

pipeline_accuracy = accuracy_score(
    y_val, pipeline_predictions
)

print("\nPIPELINE MODEL")
print("StandardScaler + Logistic Regression")
print("Validation Accuracy:", round(pipeline_accuracy * 100, 2), "%")
parameters = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_accuracy = accuracy_score(
    y_test, tuned_predictions
)

print("\nGRID SEARCH RESULTS")
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", round(grid_search.best_score_ * 100, 2), "%")
print("Tuned Test Accuracy:", round(tuned_accuracy * 100, 2), "%")
comparison = pd.DataFrame({
    'Model': [
        'Baseline Logistic Regression',
        '5-Fold Cross Validation',
        'Scaled Logistic Regression',
        'Tuned Logistic Regression'
    ],
    'Accuracy (%)': [
        round(baseline_accuracy * 100, 2),
        round(cv_scores.mean() * 100, 2),
        round(pipeline_accuracy * 100, 2),
        round(tuned_accuracy * 100, 2)
    ]
})

print("\nFINAL MODEL COMPARISON")
print(comparison.to_string(index=False))



print("\nCONCLUSION")
print("Baseline Accuracy:", round(baseline_accuracy * 100, 2), "%")
print("Tuned Accuracy:", round(tuned_accuracy * 100, 2), "%")
print("Best Hyperparameter:", grid_search.best_params_)



Dataset Loaded Successfully!

First 5 Rows:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  wo